Đang tải dữ liệu 10 năm từ Open-Meteo...

Thư mục lưu mô hình tuyệt đối: file:///d:/New folder (2)/Weather-Forecast/saved_weather_models
--------------------------------------------------
⏳ Đang huấn luyện và lưu: temperature_2m_max...
⏳ Đang huấn luyện và lưu: temperature_2m_min...
⏳ Đang huấn luyện và lưu: precipitation_sum...
⏳ Đang huấn luyện và lưu: wind_speed_10m_max...

🎉 HOÀN TẤT THÀNH CÔNG!
File CSV đã lưu tại: d:\New folder (2)\Weather-Forecast\predictions_2026.csv


In [1]:
# ==========================================
# PHẦN 1: CẤU HÌNH VÀ IMPORT THƯ VIỆN
# ==========================================
import os
import sys
import pandas as pd
import datetime
import openmeteo_requests
import requests_cache
from retry_requests import retry

# Import trực tiếp từ hệ sinh thái PySpark và PySpark MLlib (Chuẩn Big Data)
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, month, dayofyear, year, dayofweek, to_date
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor, LinearRegression, GBTRegressor

# Ép biến môi trường để PySpark tương thích với hệ điều hành cục bộ (Windows)
os.environ['HADOOP_HOME'] = 'C:\\hadoop' 
os.environ['PATH'] = os.environ.get('PATH', '') + ';C:\\hadoop\\bin'
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# Khởi tạo Spark Session
spark = (SparkSession.builder
    .appName("Weather_Training_MLlib")
    .master("local[*]")
    .config("spark.driver.memory", "16g")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

# Các trường dữ liệu mục tiêu
CORE_FIELDS = ["temperature_2m_max", "temperature_2m_min", "precipitation_sum", "wind_speed_10m_max"]
LAT, LON = 10.7626, 106.6602

# ==========================================
# PHẦN 2: THU THẬP VÀ XỬ LÝ ĐẶC TRƯNG THỜI GIAN
# ==========================================
def fetch_10_years_data(lat, lon):
    print("Đang tải dữ liệu 10 năm từ trạm quan trắc (Open-Meteo)...")
    cache_session = requests_cache.CachedSession('.cache', expire_after=3600)
    retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
    openmeteo = openmeteo_requests.Client(session=retry_session)

    url = "https://archive-api.open-meteo.com/v1/archive"
    end_date = datetime.date.today() - datetime.timedelta(days=1)
    start_date = end_date - datetime.timedelta(days=365 * 10) 

    params = {
        "latitude": lat, "longitude": lon,
        "start_date": start_date.strftime("%Y-%m-%d"),
        "end_date": end_date.strftime("%Y-%m-%d"),
        "daily": CORE_FIELDS, "timezone": "Asia/Bangkok"
    }
    responses = openmeteo.weather_api(url, params=params)
    daily = responses[0].Daily()
    
    date_range = pd.date_range(
        start=pd.to_datetime(daily.Time(), unit="s", utc=True),
        end=pd.to_datetime(daily.TimeEnd(), unit="s", utc=True),
        freq=pd.Timedelta(seconds=daily.Interval()), inclusive="left"
    )
    data_dict = {"date": date_range.tz_localize(None).astype(str)}
    for index, field in enumerate(CORE_FIELDS):
        data_dict[field] = daily.Variables(index).ValuesAsNumpy().tolist()

    return pd.DataFrame(data_dict).dropna()

# Tạo tập dữ liệu huấn luyện (Lịch sử)
pdf_history = fetch_10_years_data(LAT, LON)
df_history = spark.createDataFrame(pdf_history).withColumn("date", to_date(col("date")))

# Feature Engineering: Bóc tách thời gian để nhận diện chu kỳ mùa vụ
df_features = df_history.withColumn("year", year(col("date"))) \
                        .withColumn("month", month(col("date"))) \
                        .withColumn("day_of_year", dayofyear(col("date"))) \
                        .withColumn("day_of_week", dayofweek(col("date")))

assembler = VectorAssembler(inputCols=["year", "month", "day_of_year", "day_of_week"], outputCol="features")
ml_data = assembler.transform(df_features).cache()

# Tạo tập dữ liệu suy luận (Tương lai - Từ nay đến cuối năm)
start_of_year = datetime.date(2026, 1, 1)  # Ngày 1 tháng 1 năm 2026
end_of_year = datetime.date(2026, 12, 31)  # Ngày 31 tháng 12 năm 2026
future_dates = pd.date_range(start=start_of_year, end=end_of_year).astype(str).tolist()

pdf_future = pd.DataFrame({"date": future_dates})
df_future = spark.createDataFrame(pdf_future).withColumn("date", to_date(col("date")))
df_future = df_future.withColumn("year", year(col("date"))) \
                     .withColumn("month", month(col("date"))) \
                     .withColumn("day_of_year", dayofyear(col("date"))) \
                     .withColumn("day_of_week", dayofweek(col("date")))

future_ml_data = assembler.transform(df_future)

# ==========================================
# PHẦN 3: ĐỊNH TUYẾN ĐƯỜNG DẪN LƯU TRỮ TUYỆT ĐỐI
# ==========================================
current_dir = os.path.abspath(os.getcwd()).replace("\\", "/")
models_saved_path = f"file:///{current_dir}/saved_weather_models"

print(f"\nĐường dẫn lưu mô hình an toàn: {models_saved_path}")
print("-" * 50)

# ==========================================
# PHẦN 4: HUẤN LUYỆN, LƯU MÔ HÌNH VÀ SUY LUẬN (INFERENCE)
# ==========================================
# Cấu trúc từ điển lưu trữ 3 kết quả cho mỗi ngày
predictions_dict = {"date": future_dates}

for field in CORE_FIELDS:
    print(f"⏳ Đang xử lý huấn luyện cho đặc trưng: {field}...")
    
    # 1. Thuật toán Rừng ngẫu nhiên (Tính ổn định cao)
    rf_model = RandomForestRegressor(featuresCol="features", labelCol=field, numTrees=20).fit(ml_data)
    # 2. Thuật toán Hồi quy tuyến tính (Mức cơ sở - Baseline)
    lr_model = LinearRegression(featuresCol="features", labelCol=field).fit(ml_data)
    # 3. Thuật toán Gradient-Boosted Trees (Mô hình phức hợp thay thế XGBoost)
    gbt_model = GBTRegressor(featuresCol="features", labelCol=field, maxIter=20).fit(ml_data)
    
    # Lưu xuất mô hình thành file vật lý
    rf_model.write().overwrite().save(f"{models_saved_path}/rf_{field}")
    lr_model.write().overwrite().save(f"{models_saved_path}/lr_{field}")
    gbt_model.write().overwrite().save(f"{models_saved_path}/gbt_{field}")

    # Chạy mô hình dự đoán trên tập dữ liệu tương lai
    pred_rf = rf_model.transform(future_ml_data).select("date", col("prediction").alias(f"rf_{field}"))
    pred_lr = lr_model.transform(future_ml_data).select("date", col("prediction").alias(f"lr_{field}"))
    pred_gbt = gbt_model.transform(future_ml_data).select("date", col("prediction").alias(f"gbt_{field}"))
    
    # Chuyển đổi sang Pandas để đóng gói dữ liệu dạng bảng
    pdf_rf = pred_rf.toPandas()
    pdf_lr = pred_lr.toPandas()
    pdf_gbt = pred_gbt.toPandas()
    
    # Đưa 3 kết quả của 3 thuật toán vào cột tương ứng cho mỗi ngày
    predictions_dict[f"{field}_RandomForest"] = pdf_rf[f"rf_{field}"].round(2).tolist()
    predictions_dict[f"{field}_LinearRegression"] = pdf_lr[f"lr_{field}"].round(2).tolist()
    predictions_dict[f"{field}_GBT"] = pdf_gbt[f"gbt_{field}"].round(2).tolist()

# ==========================================
# PHẦN 5: XUẤT KẾT QUẢ CSV ĐỂ BÁO CÁO
# ==========================================
import time # Thêm thư viện thời gian

final_predictions_df = pd.DataFrame(predictions_dict)

# Gắn thêm timestamp vào tên file để chống trùng lặp và chống khóa file
file_name = f"predictions_2026_lan_chay_{int(time.time())}.csv"
csv_path = os.path.join(os.getcwd(), file_name)

final_predictions_df.to_csv(csv_path, index=False)

print("\n🎉 HOÀN TẤT QUÁ TRÌNH HUẤN LUYỆN VÀ DỰ ĐOÁN!")
print(f"File CSV chứa kết quả 3 thuật toán cho mỗi ngày đã lưu tại: {csv_path}")



# ==========================================
# PHẦN ĐÁNH GIÁ ĐỘ TIN CẬY (ĐẦY ĐỦ 4 CHỈ SỐ)
# ==========================================
import json
from pyspark.ml.evaluation import RegressionEvaluator

print("\n📊 Đang tính toán độ tin cậy của các mô hình (Đánh giá trên tập Train)...")
eval_results = {}

# Khởi tạo công cụ chấm điểm cho cả 4 chỉ số chuẩn
eval_rmse = RegressionEvaluator(predictionCol="prediction", metricName="rmse")
eval_mae = RegressionEvaluator(predictionCol="prediction", metricName="mae")
eval_r2 = RegressionEvaluator(predictionCol="prediction", metricName="r2")
eval_mse = RegressionEvaluator(predictionCol="prediction", metricName="mse")

for field in CORE_FIELDS:
    print(f" -> Đang chấm điểm cho: {field}")
    
    # Lấy kết quả dự đoán trên toàn tập dữ liệu
    pred_rf = rf_model.transform(ml_data)
    pred_lr = lr_model.transform(ml_data)
    pred_gbt = gbt_model.transform(ml_data)

    eval_results[field] = {
        "RandomForest": {
            "RMSE": round(eval_rmse.evaluate(pred_rf, {eval_rmse.labelCol: field}), 4),
            "MAE": round(eval_mae.evaluate(pred_rf, {eval_mae.labelCol: field}), 4),
            "R2": round(eval_r2.evaluate(pred_rf, {eval_r2.labelCol: field}), 4),
            "MSE": round(eval_mse.evaluate(pred_rf, {eval_mse.labelCol: field}), 4)
        },
        "LinearRegression": {
            "RMSE": round(eval_rmse.evaluate(pred_lr, {eval_rmse.labelCol: field}), 4),
            "MAE": round(eval_mae.evaluate(pred_lr, {eval_mae.labelCol: field}), 4),
            "R2": round(eval_r2.evaluate(pred_lr, {eval_r2.labelCol: field}), 4),
            "MSE": round(eval_mse.evaluate(pred_lr, {eval_mse.labelCol: field}), 4)
        },
        "XGB_GBT": {
            "RMSE": round(eval_rmse.evaluate(pred_gbt, {eval_rmse.labelCol: field}), 4),
            "MAE": round(eval_mae.evaluate(pred_gbt, {eval_mae.labelCol: field}), 4),
            "R2": round(eval_r2.evaluate(pred_gbt, {eval_r2.labelCol: field}), 4),
            "MSE": round(eval_mse.evaluate(pred_gbt, {eval_mse.labelCol: field}), 4)
        }
    }

# Lưu kết quả ra file JSON
eval_file_path = os.path.join(os.getcwd(), "model_metrics.json")
with open(eval_file_path, "w", encoding="utf-8") as f:
    json.dump(eval_results, f, ensure_ascii=False, indent=4)

print(f"🎉 Đã lưu báo cáo độ tin cậy AI (đủ 4 chỉ số) tại: {eval_file_path}")

      

Đang tải dữ liệu 10 năm từ trạm quan trắc (Open-Meteo)...

Đường dẫn lưu mô hình an toàn: file:///d:/New folder (2)/Weather-Forecast/saved_weather_models
--------------------------------------------------
⏳ Đang xử lý huấn luyện cho đặc trưng: temperature_2m_max...
⏳ Đang xử lý huấn luyện cho đặc trưng: temperature_2m_min...
⏳ Đang xử lý huấn luyện cho đặc trưng: precipitation_sum...
⏳ Đang xử lý huấn luyện cho đặc trưng: wind_speed_10m_max...

🎉 HOÀN TẤT QUÁ TRÌNH HUẤN LUYỆN VÀ DỰ ĐOÁN!
File CSV chứa kết quả 3 thuật toán cho mỗi ngày đã lưu tại: d:\New folder (2)\Weather-Forecast\predictions_2026_lan_chay_1776160340.csv

📊 Đang tính toán độ tin cậy của các mô hình (Đánh giá trên tập Train)...
 -> Đang chấm điểm cho: temperature_2m_max
 -> Đang chấm điểm cho: temperature_2m_min
 -> Đang chấm điểm cho: precipitation_sum
 -> Đang chấm điểm cho: wind_speed_10m_max
🎉 Đã lưu báo cáo độ tin cậy AI (đủ 4 chỉ số) tại: d:\New folder (2)\Weather-Forecast\model_metrics.json


In [2]:
import os
print("Thư mục thực sự mà mã đang lưu file là:")
print(os.getcwd())

Thư mục thực sự mà mã đang lưu file là:
d:\New folder (2)\Weather-Forecast
